
# 練習問題: 共役勾配法 (CG) で連立一次方程式を解く

## 目標

対称正定値 (SPD) の大きな連立一次方程式 `A x = b` を, 反復解法の定番 **共役勾配法 (CG)** で解く。CG の中身は **行列ベクトル積 (matvec)** と **内積 (reduction)** とベクトルの足し算 (axpy) の繰り返しで, どれもこれまで学んだ並列化がそのまま使える。

ここで使う `A` は **B1 (2D熱伝導) と同じ 2次元ラプラシアン行列**。同じ行列を, B1 では「定常状態を反復で求めた」のに対し, ここでは「方程式として解く」。(さらに G2 では同じ行列の固有振動モードを求める。)

## 課題

`cg.cpp` (または `cg.f90`) は, ラプラシアン行列 `A` (行列を保持せず 5点ステンシルで計算する「行列フリー」) に対し, 既知の真の解 `xt` から `b = A xt` を作り, `x=0` から CG で `A x = b` を解く。

並列化すべき2つのカーネルが `TODO` になっている:

1. **matvec** (`y = A p`): 格子点の二重ループ。
   - C++: `#pragma omp parallel for collapse(2)`
   - Fortran: `!$omp parallel do collapse(2) private(v)` … `!$omp end parallel do`
2. **dot** (内積 `a・b`): 総和。
   - C++: `#pragma omp parallel for reduction(+:s)`
   - Fortran: `!$omp parallel do reduction(+:s)` … `!$omp end parallel do`

(CG 本体のベクトル更新 axpy は逐次のまま書いてある。余力があればそこも並列化してよい。)

## コンパイルと実行

```
# C++
nvc++ -fast -mp=multicore cg.cpp -o cg.exe

# Fortran
nvfortran -fast -mp=multicore cg.f90 -o cg.exe
```

第1引数で格子の一辺 `n` (未知数は `N = n²`), 第2引数で収束しきい値。

```
OMP_NUM_THREADS=4 ./cg.exe 128 1e-8
```

## 期待される結果

```
n=128 (N=16384), iters=400, 残差=9.94e-09, 解の誤差(max|x-xt|)=3.22e-09
```

- **残差** `||A x − b||` が `tol` 未満になれば収束。
- **解の誤差** `max|x − xt|` が小さければ, 真の解を正しく復元できている (検算成功)。
- CG の反復回数はおおよそ `n` に比例する (この問題では数百回)。
- `n` を大きくする (例: `512`) と1反復が重くなり, matvec と内積の並列化の効果 (台数効果) が見えやすい。「性能を比べる」セルで測ってみよ。
- (発展) 同じ matvec を GPU にオフロードする, axpy も並列化する, などで更に速くできる。



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ cg.cpp
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <omp.h>

/* 状態を持たない乱数 (既知解の生成用): (seed,k) から [0,1)。 */
static inline double draw_rand01(long long seed, long long k) {
  const long long M = 2147483647LL;
  long long x = ((seed % M) * 2654435761LL + (k % M) + 1) % M;
  x = ((x ^ (x >> 16)) * 1812433253LL) % M;
  x = ((x ^ (x >> 13)) * 1664525LL)    % M;
  x =  (x ^ (x >> 16)) % M;
  return (double)x / (double)M;
}

/* 行列ベクトル積 y = A p。
   A は n×n 格子上の 2次元ラプラシアン (5点ステンシル, ディリクレ境界=0) で,
   対称正定値 (SPD)。行列を保持せず, ステンシルの計算で済ませる (行列フリー)。
   (A p)[i,j] = 4 p[i,j] - p[i-1,j] - p[i+1,j] - p[i,j-1] - p[i,j+1]  (領域外は 0)。
   これは B1 (2D熱伝導) と同じラプラシアン行列である。 */
void matvec(int n, const double * p, double * y) {
  // TODO: 格子点の二重ループを #pragma omp parallel for collapse(2) で並列化せよ.
  for (int i = 0; i < n; i++) {
    for (int j = 0; j < n; j++) {
      double v = 4.0 * p[i*n+j];
      if (i > 0)     v -= p[(i-1)*n+j];
      if (i < n - 1) v -= p[(i+1)*n+j];
      if (j > 0)     v -= p[i*n+j-1];
      if (j < n - 1) v -= p[i*n+j+1];
      y[i*n+j] = v;
    }
  }
}

/* 内積 a・b */
double dot(long N, const double * a, const double * b) {
  double s = 0.0;
  // TODO: 内積の和を #pragma omp parallel for reduction(+:s) で並列化せよ.
  for (long k = 0; k < N; k++) s += a[k] * b[k];
  return s;
}

int main(int argc, char ** argv) {
  int    n   = (argc > 1 ? atoi(argv[1]) : 128);    /* 格子の一辺 (未知数は N = n*n) */
  double tol = (argc > 2 ? atof(argv[2]) : 1e-8);
  long   N   = (long)n * n;
  int    maxit = 10 * n;

  double * xt = (double *)malloc(sizeof(double) * N);  /* 既知の真の解 */
  double * b  = (double *)malloc(sizeof(double) * N);
  double * x  = (double *)malloc(sizeof(double) * N);
  double * r  = (double *)malloc(sizeof(double) * N);
  double * p  = (double *)malloc(sizeof(double) * N);
  double * Ap = (double *)malloc(sizeof(double) * N);

  for (long k = 0; k < N; k++) xt[k] = draw_rand01(k, 0);  /* 真の解をランダムに決め */
  matvec(n, xt, b);                                        /* b = A xt を作る */

  /* CG: x=0 から始めて A x = b を解く */
  for (long k = 0; k < N; k++) { x[k] = 0.0; r[k] = b[k]; p[k] = b[k]; }
  double rs = dot(N, r, r);

  int it;
  double t0 = omp_get_wtime();
  for (it = 0; it < maxit; it++) {
    matvec(n, p, Ap);
    double alpha = rs / dot(N, p, Ap);
    for (long k = 0; k < N; k++) { x[k] += alpha * p[k]; r[k] -= alpha * Ap[k]; }  /* (発展: ここも並列化可) */
    double rs_new = dot(N, r, r);
    if (sqrt(rs_new) < tol) { rs = rs_new; it++; break; }
    double beta = rs_new / rs;
    for (long k = 0; k < N; k++) p[k] = r[k] + beta * p[k];
    rs = rs_new;
  }
  double elapsed = omp_get_wtime() - t0;

  /* 検算: 求めた x が真の解 xt にどれだけ近いか */
  double err = 0.0;
  for (long k = 0; k < N; k++) { double e = fabs(x[k] - xt[k]); if (e > err) err = e; }
  printf("n=%d (N=%ld), iters=%d, 残差=%.2e, 解の誤差(max|x-xt|)=%.2e\n",
         n, N, it, sqrt(rs), err);
  printf("elapsed = %.3f sec\n", elapsed);
  free(xt); free(b); free(x); free(r); free(p); free(Ap);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore cg.cpp -o cg_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./cg_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ cg.f90
module cg_mod
contains
  ! 状態を持たない乱数 (既知解の生成用): (seed,k) から [0,1)。
  function draw_rand01(seed, k) result(u)
    integer(8), intent(in) :: seed, k
    real(8) :: u
    integer(8), parameter :: M = 2147483647_8
    integer(8) :: x
    x = modulo(modulo(seed, M) * 2654435761_8 + modulo(k, M) + 1_8, M)
    x = modulo(ieor(x, ishft(x, -16)) * 1812433253_8, M)
    x = modulo(ieor(x, ishft(x, -13)) * 1664525_8,    M)
    x = modulo(ieor(x, ishft(x, -16)), M)
    u = real(x, 8) / real(M, 8)
  end function draw_rand01

  ! 行列ベクトル積 y = A p。A は n×n 格子の 2次元ラプラシアン (5点ステンシル,
  ! ディリクレ境界=0) で対称正定値。行列を保持せずステンシルで計算する (行列フリー)。
  ! これは B1 (2D熱伝導) と同じラプラシアン行列。添字は 0..n*n-1 (idx = i*n + j)。
  subroutine matvec(n, p, y)
    integer, intent(in) :: n
    real(8), intent(in) :: p(0:n*n-1)
    real(8), intent(out) :: y(0:n*n-1)
    integer :: i, j
    real(8) :: v
    ! TODO: 格子点の二重ループを !$omp parallel do collapse(2) private(v) で並列化せよ.
    do i = 0, n - 1
       do j = 0, n - 1
          v = 4.0d0 * p(i*n+j)
          if (i > 0)     v = v - p((i-1)*n+j)
          if (i < n - 1) v = v - p((i+1)*n+j)
          if (j > 0)     v = v - p(i*n+j-1)
          if (j < n - 1) v = v - p(i*n+j+1)
          y(i*n+j) = v
       end do
    end do
    ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do)。
  end subroutine matvec

  ! 内積 a・b
  function dot(N, a, b) result(s)
    integer(8), intent(in) :: N
    real(8), intent(in) :: a(0:N-1), b(0:N-1)
    real(8) :: s
    integer(8) :: k
    s = 0.0d0
    ! TODO: 内積の和を !$omp parallel do reduction(+:s) で並列化せよ.
    do k = 0, N - 1
       s = s + a(k) * b(k)
    end do
    ! TODO: 上で始めた parallel do 領域を閉じる (!$omp end parallel do)。
  end function dot
end module cg_mod

program cg
  use cg_mod
  use omp_lib
  character(len=32) :: arg
  integer :: n, maxit, it
  real(8) :: tol, rs, rs_new, alpha, beta, err, e, t0, elapsed
  integer(8) :: N8, k
  real(8), allocatable :: xt(:), b(:), x(:), r(:), p(:), Ap(:)
  n = 128; tol = 1d-8
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) n
  end if
  if (command_argument_count() >= 2) then
     call get_command_argument(2, arg); read (arg, *) tol
  end if
  N8 = int(n, 8) * n
  maxit = 10 * n

  allocate(xt(0:N8-1), b(0:N8-1), x(0:N8-1), r(0:N8-1), p(0:N8-1), Ap(0:N8-1))
  do k = 0, N8 - 1
     xt(k) = draw_rand01(k, 0_8)        ! 真の解をランダムに決め
  end do
  call matvec(n, xt, b)                 ! b = A xt を作る

  ! CG: x=0 から始めて A x = b を解く
  do k = 0, N8 - 1
     x(k) = 0.0d0; r(k) = b(k); p(k) = b(k)
  end do
  rs = dot(N8, r, r)

  t0 = omp_get_wtime()
  do it = 1, maxit
     call matvec(n, p, Ap)
     alpha = rs / dot(N8, p, Ap)
     do k = 0, N8 - 1
        x(k) = x(k) + alpha * p(k); r(k) = r(k) - alpha * Ap(k)   ! (発展: ここも並列化可)
     end do
     rs_new = dot(N8, r, r)
     if (sqrt(rs_new) < tol) then
        rs = rs_new; exit
     end if
     beta = rs_new / rs
     do k = 0, N8 - 1
        p(k) = r(k) + beta * p(k)
     end do
     rs = rs_new
  end do
  elapsed = omp_get_wtime() - t0

  ! 検算: 求めた x が真の解 xt にどれだけ近いか
  err = 0.0d0
  do k = 0, N8 - 1
     e = abs(x(k) - xt(k)); if (e > err) err = e
  end do
  print "(a,i0,a,i0,a,i0,a,es9.2,a,es9.2)", &
       "n=", n, " (N=", N8, "), iters=", min(it, maxit), &
       ", 残差=", sqrt(rs), ", 解の誤差(max|x-xt|)=", err
  print "(a,f0.3,a)", "elapsed = ", elapsed, " sec"
end program cg

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore cg.f90 -o cg_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./cg_f90.exe


# 4. 発展目標 (できる範囲で挑戦)
- この問題の基本は **マルチコア並列化** (`#pragma omp parallel for` / `!$omp parallel do` など)。まずはここまでを目指す。
- 余力があれば次にも挑戦:
  - **GPUで並列化**: コンパイルを `-mp=gpu` にして, 重いループに `#pragma omp target teams distribute parallel for` (+ 必要に応じて `map`) を付ける。
  - **SIMD化** (11/12章): 内側ループに `#pragma omp simd`, またはベクトル型。 **ILP向上** (13章): ベクトル長 `nl` の調整。
- どこまで速くできるか挑戦し, その効果を下の「性能を比べる」で可視化しよう。

# 5. 性能を比べる (任意)
- 各プログラムは主計算部分の所要時間を `elapsed = ... sec` の形で表示する。
- 構成を変えて (スレッド数, SIMDの有無, GPU など) 実行し, 得られた **経過秒** を下の `DATA` に「ラベルと秒」で並べて実行すると, 棒グラフと「基準(先頭)比のスピードアップ」が出る。
- 試した構成だけ書けばよい (`#` で始まる行は無視)。


In [ ]:
import matplotlib.pyplot as plt

# 各構成の (ラベル, 経過秒) を並べる。先頭の行を基準(スピードアップ=1)にする。
# 自分が実際に試した構成の数値に書き換える。
DATA = [
    ("serial",    1.00),
    ("omp-8",     0.20),
    # ("simd",      0.50),
    # ("simd+omp",  0.07),
    # ("gpu",       0.05),
]

labels = [d[0] for d in DATA]
secs   = [float(d[1]) for d in DATA]
speed  = [secs[0] / s for s in secs]                 # 先頭(基準)比のスピードアップ
fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].bar(labels, secs)
ax[0].set_ylabel("elapsed (s)")
ax[0].set_title("time (lower = faster)")
ax[1].bar(labels, speed)
ax[1].set_ylabel("speedup vs " + labels[0])
ax[1].set_title("speedup (higher = faster)")
for a in ax:
    a.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.show()


# 6. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:cg.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 6-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 6-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 6-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:cg.cpp}

コマンドと実行結果:
{bash[-1]}

## 6-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:cg.cpp}